# GIS4: 상권 기반 Hub & Spoke 물류 전략

## 분석 목적
- 단기체류 외국인 수요와 관광 이동 신호를 이용해 점포별 재고 배분을 최적화합니다.
- 장기체류 외국인은 내국인 소비패턴과 유사하다고 보고 본 모델에서 제외합니다.
- `Hub_A / Hub_B / Normal / Spoke` 운영 정책과 재고 배율을 설계합니다.

## 사용 데이터
- `data/pop/seoul_250M_population_foreigner_shortterm.csv`
- `data/pop/스마트서울 도시데이터 센서(S-DoT) 유동인구 측정 정보.csv`
- `data/pop/서울교통공사_1회권외국인일별통행통계.csv`
- `data/pop/서울교통공사_외국인 관광객 기간권 일별통행통계.csv`
- `data/pop/match/match.shp`
- `data/mall_info/daiso.csv`
- `data/mall_info/mall_info_seoul.csv` (올리브영 거리 계산)


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
from IPython.display import display

PROJECT_ROOT = Path(r"G:/Final_proj/Total_clear/데이터")
DATA_DIR = PROJECT_ROOT / "data"
DATA = DATA_DIR
OUTPUT_DIR = PROJECT_ROOT / "3.GIS_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TARGET_CRS = "EPSG:5186"


def clean_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace(",", "", regex=False).replace("*", np.nan),
        errors="coerce",
    )


def zscore(s):
    s = s.fillna(0)
    std = s.std(ddof=0)
    return (s - s.mean()) / (std if std != 0 else 1)


def u(unicode_escape_text: str) -> str:
    return unicode_escape_text.encode("utf-8").decode("unicode_escape")


d:\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## 1) 점포 데이터 로드 및 주소 기반 자치구 파싱

In [ ]:

# 다이소 점포 목록 로드
stores_df = pd.read_csv(DATA / "mall_info" / "daiso.csv", encoding="utf-8-sig").iloc[:, :4].copy()
stores_df.columns = ["store_name", "lat", "lon", "addr"]
stores_df["lat"] = pd.to_numeric(stores_df["lat"], errors="coerce")
stores_df["lon"] = pd.to_numeric(stores_df["lon"], errors="coerce")
stores_df = stores_df.dropna(subset=["lat", "lon"]).drop_duplicates(subset=["store_name", "lat", "lon"]).reset_index(drop=True)
stores_df["store_id"] = np.arange(len(stores_df))

# 주소 토큰에서 자치구(구) 추출

def get_gu(addr):
    toks = str(addr).split()
    if len(toks) >= 2:
        cand = toks[1].strip()
        if cand and ord(cand[-1]) == 0xAD6C:  # "구"
            return cand
    return np.nan

stores_df["gu_kor"] = stores_df["addr"].apply(get_gu)

kor_to_eng = {
    u("\\uc885\\ub85c\\uad6c"): "Jongno-gu",
    u("\\uc911\\uad6c"): "Jung-gu",
    u("\\uc6a9\\uc0b0\\uad6c"): "Yongsan-gu",
    u("\\uc131\\ub3d9\\uad6c"): "Seongdong-gu",
    u("\\uad11\\uc9c4\\uad6c"): "Gwangjin-gu",
    u("\\ub3d9\\ub300\\ubb38\\uad6c"): "Dongdaemun-gu",
    u("\\uc911\\ub791\\uad6c"): "Jungnang-gu",
    u("\\uc131\\ubd81\\uad6c"): "Seongbuk-gu",
    u("\\uac15\\ubd81\\uad6c"): "Gangbuk-gu",
    u("\\ub3c4\\ubd09\\uad6c"): "Dobong-gu",
    u("\\ub178\\uc6d0\\uad6c"): "Nowon-gu",
    u("\\uc740\\ud3c9\\uad6c"): "Eunpyeong-gu",
    u("\\uc11c\\ub300\\ubb38\\uad6c"): "Seodaemun-gu",
    u("\\ub9c8\\ud3ec\\uad6c"): "Mapo-gu",
    u("\\uc591\\ucc9c\\uad6c"): "Yangcheon-gu",
    u("\\uac15\\uc11c\\uad6c"): "Gangseo-gu",
    u("\\uad6c\\ub85c\\uad6c"): "Guro-gu",
    u("\\uae08\\ucc9c\\uad6c"): "Geumcheon-gu",
    u("\\uc601\\ub4f1\\ud3ec\\uad6c"): "Yeongdeungpo-gu",
    u("\\ub3d9\\uc791\\uad6c"): "Dongjak-gu",
    u("\\uad00\\uc545\\uad6c"): "Gwanak-gu",
    u("\\uc11c\\ucd08\\uad6c"): "Seocho-gu",
    u("\\uac15\\ub0a8\\uad6c"): "Gangnam-gu",
    u("\\uc1a1\\ud30c\\uad6c"): "Songpa-gu",
    u("\\uac15\\ub3d9\\uad6c"): "Gangdong-gu",
}
stores_df["gu"] = stores_df["gu_kor"].map(kor_to_eng)

stores = gpd.GeoDataFrame(
    stores_df,
    geometry=gpd.points_from_xy(stores_df["lon"], stores_df["lat"]),
    crs="EPSG:4326",
).to_crs(TARGET_CRS)

print("점포 수:", len(stores), "자치구 미매칭 수:", stores["gu"].isna().sum())


점포 수: 231 자치구 미매칭 수: 0


## 2) 올리브영 50~100m 경쟁 플래그 생성 (GIS1 로직 반영)

In [ ]:

mall = pd.read_csv(DATA / "mall_info" / "mall_info_seoul.csv", low_memory=False).iloc[:, [1, 38, 37]].copy()
mall.columns = ["name", "lat", "lon"]

kw_olive = "".join(map(chr, [0xC62C, 0xB9AC, 0xBE0C, 0xC601]))
olive = mall[mall["name"].astype(str).str.contains(kw_olive, na=False)].copy()
olive["lat"] = pd.to_numeric(olive["lat"], errors="coerce")
olive["lon"] = pd.to_numeric(olive["lon"], errors="coerce")
olive = olive.dropna(subset=["lat", "lon"]).reset_index(drop=True)

olive_g = gpd.GeoDataFrame(
    olive,
    geometry=gpd.points_from_xy(olive["lon"], olive["lat"]),
    crs="EPSG:4326",
).to_crs(TARGET_CRS)

near = gpd.sjoin_nearest(
    stores[["store_id", "geometry"]],
    olive_g[["geometry"]],
    how="left",
    distance_col="dist_olive_m",
)
near = near[["store_id", "dist_olive_m"]].drop_duplicates("store_id")
stores = stores.merge(near, on="store_id", how="left")
stores["comp_50_100"] = ((stores["dist_olive_m"] >= 50) & (stores["dist_olive_m"] <= 100)).astype(int)

print("올리브영 50~100m 경쟁 점포 수:", int(stores["comp_50_100"].sum()))


올리브영 50~100m 경쟁 점포 수: 35


## 3) 점포 반경 500m 내 단기체류 외국인 250m 격자 집계

In [ ]:

fs = pd.read_csv(DATA / "pop" / "seoul_250M_population_foreigner_shortterm.csv", encoding="cp949", low_memory=False).iloc[:, :24].copy()
num_cols = fs.columns[5:].tolist()  # detailed national columns (exclude metadata + aggregate)
for c in num_cols:
    fs[c] = clean_num(fs[c])

fs["foreign_short_total"] = fs[num_cols].sum(axis=1)
cell_col = fs.columns[3]
fs_cell = (
    fs.groupby(cell_col, as_index=False)["foreign_short_total"]
      .mean()
      .rename(columns={cell_col: "CELL_ID", "foreign_short_total": "fs_cell_mean"})
)
fs_cell["CELL_ID"] = fs_cell["CELL_ID"].astype(str).str.strip()

grid = gpd.read_file(DATA / "pop" / "match" / "match.shp")
if grid.crs is None:
    grid = grid.set_crs(TARGET_CRS)
else:
    grid = grid.to_crs(TARGET_CRS)

grid["CELL_ID"] = grid["CELL_ID"].astype(str).str.strip()
grid = grid.merge(fs_cell, on="CELL_ID", how="left")
grid["fs_cell_mean"] = grid["fs_cell_mean"].fillna(0)

cent = gpd.GeoDataFrame(
    grid[["CELL_ID", "fs_cell_mean"]].copy(),
    geometry=grid.geometry.centroid,
    crs=grid.crs,
)

buf = stores[["store_id", "geometry"]].copy()
buf["geometry"] = buf.geometry.buffer(500)
join = gpd.sjoin(cent[["fs_cell_mean", "geometry"]], buf, how="inner", predicate="within")
agg = join.groupby("store_id")["fs_cell_mean"].sum().rename("foreign_500m_sum").reset_index()
stores = stores.merge(agg, on="store_id", how="left")
stores["foreign_500m_sum"] = stores["foreign_500m_sum"].fillna(0)

date_col = fs.columns[0]
print("단기체류 외국인 데이터 기간:", fs[date_col].astype(str).min(), "~", fs[date_col].astype(str).max(), "(일자 개수=", fs[date_col].nunique(), ")")


단기체류 외국인 데이터 기간: 20260129 ~ 20260129 (일자 개수= 1 )


## 4) S-DoT 자치구 유동 + 외국인 지하철 이용 추세 결합

In [ ]:

# S-DoT 데이터
sdot_file = [f for f in (DATA / "pop").iterdir() if f.is_file() and f.suffix.lower() == ".csv" and "S-DoT" in f.name][0]
sdot = pd.read_csv(sdot_file, encoding="cp949", low_memory=False).iloc[:, [2, 4, 6]].copy()
sdot.columns = ["meas_time", "gu", "visitors"]
sdot["visitors"] = clean_num(sdot["visitors"]).fillna(0)
sdot["gu"] = sdot["gu"].astype(str).str.strip()
sdot_gu = sdot.groupby("gu", as_index=False)["visitors"].mean().rename(columns={"visitors": "sdot_gu_mean"})

stores = stores.merge(sdot_gu, on="gu", how="left")
stores["sdot_gu_mean"] = stores["sdot_gu_mean"].fillna(stores["sdot_gu_mean"].median())

# 지하철 데이터(컬럼 수 기준 자동 선택)
sub_candidates = []
for f in (DATA / "pop").iterdir():
    if f.is_file() and f.suffix.lower() == ".csv" and "seoul_250M_population" not in f.name and "S-DoT" not in f.name:
        d = pd.read_csv(f, encoding="cp949", nrows=2)
        sub_candidates.append((f, len(d.columns)))

one_file = [x[0] for x in sub_candidates if x[1] == 31][0]  # one-ticket
pass_file = [x[0] for x in sub_candidates if x[1] == 10][0]  # tourist pass

sub1 = pd.read_csv(one_file, encoding="cp949", low_memory=False)
hour_cols = sub1.columns[7:31]
for c in hour_cols:
    sub1[c] = clean_num(sub1[c]).fillna(0)
sub1["cnt"] = sub1[hour_cols].sum(axis=1)
station_col = sub1.columns[3]
sub1_station = sub1.groupby(station_col, as_index=False)["cnt"].sum().rename(columns={station_col: "station", "cnt": "subway_1ticket"})

sub2 = pd.read_csv(pass_file, encoding="cp949", low_memory=False)
station2_col = sub2.columns[4]
board_col = sub2.columns[8]
alight_col = sub2.columns[9]
sub2[board_col] = clean_num(sub2[board_col]).fillna(0)
sub2[alight_col] = clean_num(sub2[alight_col]).fillna(0)
sub2["cnt"] = sub2[board_col] + sub2[alight_col]
sub2_station = sub2.groupby(station2_col, as_index=False)["cnt"].sum().rename(columns={station2_col: "station", "cnt": "subway_pass"})

station = sub1_station.merge(sub2_station, on="station", how="outer").fillna(0)
station["station_score"] = zscore(station["subway_1ticket"]) + zscore(station["subway_pass"])
station = station.sort_values("station_score", ascending=False)

# 역명-점포명 단순 매칭
stores["station_match"] = ""
stores["subway_station_score"] = 0.0
pairs = [(str(r.station).replace(" ", ""), float(r.station_score), str(r.station)) for r in station.itertuples(index=False)]
for i, r in stores.iterrows():
    s = str(r["store_name"]).replace(" ", "")
    hits = [p for p in pairs if p[0] and p[0] in s]
    if hits:
        best = max(hits, key=lambda x: x[1])
        stores.at[i, "station_match"] = best[2]
        stores.at[i, "subway_station_score"] = best[1]


## 5) Hub/Spoke 점수화 및 재고 배율 정책

In [ ]:

stores["z_foreign500"] = zscore(stores["foreign_500m_sum"])
stores["z_sdot"] = zscore(stores["sdot_gu_mean"])
stores["z_subway"] = zscore(stores["subway_station_score"])
stores["z_comp"] = zscore(stores["comp_50_100"])

# 가중 점수: 단기 외국인 수요 중심
stores["hub_score"] = (
    0.55 * stores["z_foreign500"]
    + 0.20 * stores["z_sdot"]
    + 0.15 * stores["z_subway"]
    + 0.10 * stores["z_comp"]
)

q90 = stores["hub_score"].quantile(0.90)
q70 = stores["hub_score"].quantile(0.70)
q30 = stores["hub_score"].quantile(0.30)

stores["role"] = np.select(
    [stores["hub_score"] >= q90, stores["hub_score"] >= q70, stores["hub_score"] < q30],
    ["Hub_A", "Hub_B", "Spoke"],
    default="Normal",
)

rank_pct = stores["hub_score"].rank(pct=True)
stores["inv_multiplier"] = np.select(
    [stores["role"].eq("Hub_A"), stores["role"].eq("Hub_B"), stores["role"].eq("Spoke")],
    [
        2.0 + (rank_pct - 0.9).clip(lower=0) / 0.1 * 0.6,  # 2.0~2.6x
        1.3 + (rank_pct - 0.7).clip(lower=0) / 0.2 * 0.5,  # 1.3~1.8x
        0.8,
    ],
    default=1.0,
)

role_cnt = stores["role"].value_counts()
print(role_cnt)


role
Normal    92
Spoke     69
Hub_B     46
Hub_A     24
Name: count, dtype: int64


## 5-1) 재고 배율 산정 근거

아래 수치는 현재 데이터(단기체류 외국인 2026-01-29 단일 기준)으로 산출한 값입니다.

### 1) 점수 공식
- `hub_score = 0.55*z(foreign_500m_sum) + 0.20*z(sdot_gu_mean) + 0.15*z(subway_station_score) + 0.10*z(comp_50_100)`

### 2) 분위수 임계값(실제 계산)
- P30: **-0.2617**
- P50: **-0.1138**
- P70: **0.0254**
- P90: **0.3760**
- P95: **0.7113**

즉, `P90 이상 = Hub_A`, `P70~P90 = Hub_B`, `P30 미만 = Spoke`로 역할이 나뉘니다.

### 3) “몇 배”의 숫자 근거 (수요 대비)
`hub_score` 구간별 점포 주변 단기외국인 수요 중앙값은 다음과 같습니다.

- P50~70 구간 중앙값: **137.63** (기준구간)
- P70~90 구간 중앙값: **323.81**  -> 기준 대비 **2.35배**
- P90~100 구간 중앙값: **939.15** -> 기준 대비 **6.82배**
- P0~30 구간 중앙값: **90.43**  -> 기준 대비 **0.66배**

이 수요차를 그대로 적용하면 과쟉재고 리스크가 크기 때문에, 운영 캡을 두어 실행 배율을 다음과 같이 정의했습니다.

- **Hub_A: 2.0~2.6x** (상위 10% 구간, 품절 방어 우선)
- **Hub_B: 1.3~1.8x** (상위 30~10% 구간, 수요 증가 흡수)
- **Normal: 1.0x**
- **Spoke: 0.8x** (하위 30% 구간, 과쟉재고 방지)

### 4) 실제 그룹별 집계(현재 결과)
| Role | 점포수 | 단기외국인 중앙값(500m) | Normal 대비 비율 | 적용 배율 범위 |
|---|---:|---:|---:|---:|
| Hub_A | 24 | 936.46 | 7.02x | 2.00~2.60x |
| Hub_B | 46 | 316.04 | 2.37x | 1.30~1.79x |
| Normal | 92 | 133.49 | 1.00x | 1.00x |
| Spoke | 69 | 90.44 | 0.68x | 0.80x |

> 요약: 배율은 “자의적 설정”이 아니라, `hub_score 분위수`와 `구간별 실제 수요차`를 숫자로 고정한 운영 규칙입니다.


## 5-2) 배율 그리드서치(총재고 고정 + 운영 제약)
- 목적: `Hub_A 2배 내외`가 왜 나오는지 숫자로 검증
- 방식: 수요 프록시(`foreign_500m_sum`)를 기준으로 배율 조합 전수 탐색
- 제약: Hub_A 집중도 상한, Spoke/Normal 최소 재고 비중


In [ ]:
if "u" not in globals():
    def u(unicode_escape_text: str) -> str:
        return unicode_escape_text.encode("utf-8").decode("unicode_escape")


demand = stores["foreign_500m_sum"].clip(lower=0).to_numpy(dtype=float)
roles = stores["role"].to_numpy()
n_store = len(demand)
total_demand = float(demand.sum())


def equal_fill_from_ratio(stock_ratio: float) -> float:
    stock = np.full(n_store, stock_ratio * total_demand / n_store, dtype=float)
    sold = np.minimum(demand, stock).sum()
    return float(sold / total_demand) if total_demand > 0 else 0.0


def calibrate_total_stock(target_fill: float = 0.55) -> float:
    lo, hi = 0.01, 5.0
    for _ in range(80):
        mid = (lo + hi) / 2
        if equal_fill_from_ratio(mid) < target_fill:
            lo = mid
        else:
            hi = mid
    ratio = (lo + hi) / 2
    return float(ratio * total_demand), float(ratio)


def eval_const_policy(m_a: float, m_b: float, m_s: float, total_stock: float) -> dict:
    m = np.ones(n_store, dtype=float)
    m[roles == "Hub_A"] = m_a
    m[roles == "Hub_B"] = m_b
    m[roles == "Spoke"] = m_s

    alloc = total_stock * m / m.sum()
    sold = np.minimum(demand, alloc)
    unmet = float((demand - sold).clip(min=0).sum())
    over = float((alloc - demand).clip(min=0).sum())
    fill = float(sold.sum() / total_demand) if total_demand > 0 else 0.0

    out = {
        "mA": float(m_a),
        "mB": float(m_b),
        "mS": float(m_s),
        "fill": fill,
        "unmet": unmet,
        "over": over,
    }

    for r in ["Hub_A", "Hub_B", "Normal", "Spoke"]:
        mask = roles == r
        role_d = float(demand[mask].sum())
        role_s = float(sold[mask].sum())
        role_alloc = float(alloc[mask].sum())
        out[f"fill_{r}"] = (role_s / role_d) if role_d > 0 else np.nan
        out[f"stock_share_{r}"] = (role_alloc / total_stock) if total_stock > 0 else np.nan

    return out


def eval_current_rank_policy(total_stock: float) -> dict:
    rank_pct = stores["hub_score"].rank(pct=True).to_numpy()
    m = np.ones(n_store, dtype=float)
    mask_a = roles == "Hub_A"
    mask_b = roles == "Hub_B"
    mask_s = roles == "Spoke"

    m[mask_a] = 2.0 + np.clip(rank_pct[mask_a] - 0.9, 0, None) / 0.1 * 0.6
    m[mask_b] = 1.3 + np.clip(rank_pct[mask_b] - 0.7, 0, None) / 0.2 * 0.5
    m[mask_s] = 0.8

    alloc = total_stock * m / m.sum()
    sold = np.minimum(demand, alloc)
    unmet = float((demand - sold).clip(min=0).sum())
    over = float((alloc - demand).clip(min=0).sum())
    fill = float(sold.sum() / total_demand) if total_demand > 0 else 0.0

    out = {
        "mA_mean": float(m[mask_a].mean()) if mask_a.any() else np.nan,
        "mB_mean": float(m[mask_b].mean()) if mask_b.any() else np.nan,
        "mS_mean": float(m[mask_s].mean()) if mask_s.any() else np.nan,
        "fill": fill,
        "unmet": unmet,
        "over": over,
    }

    for r in ["Hub_A", "Hub_B", "Normal", "Spoke"]:
        mask = roles == r
        role_d = float(demand[mask].sum())
        role_s = float(sold[mask].sum())
        role_alloc = float(alloc[mask].sum())
        out[f"fill_{r}"] = (role_s / role_d) if role_d > 0 else np.nan
        out[f"stock_share_{r}"] = (role_alloc / total_stock) if total_stock > 0 else np.nan

    return out


TOTAL_STOCK, alpha_ratio = calibrate_total_stock(target_fill=0.55)
current = eval_current_rank_policy(TOTAL_STOCK)

rows = []
for m_a in np.round(np.arange(1.6, 3.41, 0.1), 2):
    for m_b in np.round(np.arange(1.1, min(m_a, 2.5) + 1e-9, 0.1), 2):
        for m_s in np.round(np.arange(0.7, 1.01, 0.05), 2):
            rows.append(eval_const_policy(m_a, m_b, m_s, TOTAL_STOCK))

grid_df = pd.DataFrame(rows)

best_unconstrained = grid_df.sort_values("fill", ascending=False).head(1).iloc[0]

HUBA_SHARE_MAX = 0.22
SPOKE_SHARE_MIN = 0.18
NORMAL_SHARE_MIN = 0.30

feasible = grid_df[
    (grid_df["stock_share_Hub_A"] <= HUBA_SHARE_MAX)
    & (grid_df["stock_share_Spoke"] >= SPOKE_SHARE_MIN)
    & (grid_df["stock_share_Normal"] >= NORMAL_SHARE_MIN)
].copy()

if feasible.empty:
    raise ValueError("\uc6b4\uc601 \uc81c\uc57d\uc744 \ub9cc\uc871\ud558\ub294 \ud6c4\ubcf4\uac00 \uc5c6\uc2b5\ub2c8\ub2e4. \uc81c\uc57d\uc744 \uc644\ud654\ud558\uc138\uc694.")

best_const = feasible.sort_values("fill", ascending=False).head(1).iloc[0]
near_best = feasible[feasible["fill"] >= best_const["fill"] - 0.002].copy()

stores["inv_multiplier_grid"] = np.select(
    [stores["role"].eq("Hub_A"), stores["role"].eq("Hub_B"), stores["role"].eq("Spoke")],
    [best_const["mA"], best_const["mB"], best_const["mS"]],
    default=1.0,
)

summary_df = pd.DataFrame([
    {
        "policy": "current_rank",
        "mA": current["mA_mean"],
        "mB": current["mB_mean"],
        "mS": current["mS_mean"],
        "fill": current["fill"],
        "unmet": current["unmet"],
        "shareA": current["stock_share_Hub_A"],
        "shareS": current["stock_share_Spoke"],
        "shareN": current["stock_share_Normal"],
    },
    {
        "policy": "best_unconstrained",
        "mA": best_unconstrained["mA"],
        "mB": best_unconstrained["mB"],
        "mS": best_unconstrained["mS"],
        "fill": best_unconstrained["fill"],
        "unmet": best_unconstrained["unmet"],
        "shareA": best_unconstrained["stock_share_Hub_A"],
        "shareS": best_unconstrained["stock_share_Spoke"],
        "shareN": best_unconstrained["stock_share_Normal"],
    },
    {
        "policy": "best_constrained",
        "mA": best_const["mA"],
        "mB": best_const["mB"],
        "mS": best_const["mS"],
        "fill": best_const["fill"],
        "unmet": best_const["unmet"],
        "shareA": best_const["stock_share_Hub_A"],
        "shareS": best_const["stock_share_Spoke"],
        "shareN": best_const["stock_share_Normal"],
    },
])

print("[\uadf8\ub9ac\ub4dc\uc11c\uce58 \uc124\uc815]")
print(f"- {'\ubaa9\ud45c \uae30\uc900(\uade0\ub4f1\ubc30\ubd84)'}: {55.0:.1f}%")
print(f"- {'\ucd1d\uc7ac\uace0 \uce98\ub9ac\ube0c\ub808\uc774\uc158 \ube44\uc728(alpha)'}: {alpha_ratio:.4f}")
print(f"- {'\uac80\uc0c9 \ud6c4\ubcf4 \uc218'}: {len(grid_df):,}")
print()

print("[\ubb34\uc81c\uc57d \ucd5c\uc801]")
print(
    f"- mA={best_unconstrained['mA']:.2f}, mB={best_unconstrained['mB']:.2f}, mS={best_unconstrained['mS']:.2f} | "
    f"fill={best_unconstrained['fill']*100:.2f}%"
)
print()

print("[\uc6b4\uc601\uc81c\uc57d \ucd5c\uc801]")
print(
    f"- mA={best_const['mA']:.2f}, mB={best_const['mB']:.2f}, mS={best_const['mS']:.2f} | "
    f"fill={best_const['fill']*100:.2f}%"
)
print(
    f"- {'\uc81c\uc57d'}: "
    f"Hub_A<={HUBA_SHARE_MAX:.0%}, Spoke>={SPOKE_SHARE_MIN:.0%}, Normal>={NORMAL_SHARE_MIN:.0%}"
)
print(
    f"- {'\uadfc\uc811 \ucd5c\uc801 \uad6c\uac04(\uc0c1\uc704 0.2%p)'}: "
    f"mA[{near_best['mA'].min():.2f}~{near_best['mA'].max():.2f}], "
    f"mB[{near_best['mB'].min():.2f}~{near_best['mB'].max():.2f}], "
    f"mS[{near_best['mS'].min():.2f}~{near_best['mS'].max():.2f}]"
)
print()

print("[\ud604\ud589 rank \uc815\ucc45 vs \uc6b4\uc601\uc81c\uc57d \ucd5c\uc801 \ube44\uad50]")
display(summary_df)

show_cols = [
    "mA", "mB", "mS", "fill", "unmet",
    "stock_share_Hub_A", "stock_share_Spoke", "stock_share_Normal",
    "fill_Hub_A", "fill_Hub_B", "fill_Normal", "fill_Spoke",
]

print("[\uc6b4\uc601\uc81c\uc57d \ucda9\uc871 \ud6c4\ubcf4 Top 10]")
display(feasible.sort_values("fill", ascending=False).head(10)[show_cols])


[그리드서치 설정]
- 목표 기준(균등배분): 55.0%
- 총재고 캘리브레이션 비율(alpha): 0.9985
- 검색 후보 수: 1,680

[무제약 최적]
- mA=3.40, mB=1.70, mS=0.70 | fill=67.54%

[운영제약 최적]
- mA=2.80, mB=2.00, mS=0.80 | fill=65.27%
- 제약: Hub_A<=22%, Spoke>=18%, Normal>=30%
- 근접 최적 구간(상위 0.2%p): mA[2.60~2.80], mB[1.70~2.00], mS[0.75~0.80]

[현행 rank 정책 vs 운영제약 최적 비교]


,policy,mA,mB,mS,fill,unmet,shareA,shareS,shareN
0,current_rank,2.301299,1.546753,0.8,0.646534,26790.988148,0.201882,0.201768,0.336280
1,best_unconstrained,3.400000,1.700000,0.7,0.675375,24604.986348,0.271909,0.160946,0.306564
2,best_constrained,2.800000,2.000000,0.8,0.652721,26322.048014,0.219321,0.180157,0.300261


[운영제약 충족 후보 Top 10]


,mA,mB,mS,fill,unmet,stock_share_Hub_A,stock_share_Spoke,stock_share_Normal,fill_Hub_A,fill_Hub_B,fill_Normal,fill_Spoke
1010,2.8,2.0,0.80,0.652721,26322.048014,0.219321,0.180157,0.300261,0.423236,0.842997,0.829131,0.907835
778,2.6,1.7,0.75,0.651696,26399.717490,0.219448,0.181994,0.323545,0.423418,0.815194,0.851186,0.909586
891,2.7,1.8,0.80,0.651559,26410.119214,0.219810,0.187246,0.312076,0.423939,0.821642,0.840785,0.913889
898,2.7,1.9,0.80,0.650741,26472.111040,0.216433,0.184369,0.307281,0.419079,0.833809,0.836321,0.911532
905,2.7,2.0,0.80,0.649868,26538.267701,0.213158,0.181579,0.302632,0.414367,0.845608,0.831646,0.909194
779,2.6,1.7,0.80,0.649244,26585.568410,0.216817,0.191800,0.319666,0.419633,0.811491,0.847694,0.917620
892,2.7,1.8,0.85,0.649193,26589.438130,0.217267,0.196647,0.308466,0.420280,0.818064,0.837460,0.921592
666,2.5,1.6,0.75,0.648989,26604.951077,0.216333,0.186587,0.331711,0.418936,0.803708,0.857975,0.913349
659,2.5,1.5,0.75,0.648541,26638.903664,0.219982,0.189734,0.337305,0.424186,0.784700,0.862501,0.915928
786,2.6,1.8,0.80,0.648481,26643.448586,0.213406,0.188782,0.314637,0.414724,0.824180,0.843117,0.915148


### 5-3) 해석: 왜 Hub_A 배율이 7배가 아닌 2배대인가
- `7배`는 점포 주변 단기외국인 수요 비율이고, 실행 재고배율과는 다릅니다.
- 동일 총재고 조건에서 운영 제약(Hub_A 집중도 상한, Spoke/Normal 최소 비중)을 걸면 후보가 2.6~2.8대로 수렴합니다.
- 즉, `2배대`는 단순 감이 아니라 `수요 추종 + 네트워크 리스크 방어`를 동시에 만족시키는 타협 해입니다.


## 6) 결과 확인: Hub 후보 / Spoke 후보 / 핵심 지표

In [ ]:

out_cols = [
    "store_name", "addr", "gu", "foreign_500m_sum", "sdot_gu_mean",
    "station_match", "subway_station_score", "dist_olive_m", "hub_score", "role", "inv_multiplier"
]

print("[상위 Hub 15개]")
display(stores.sort_values("hub_score", ascending=False)[out_cols].head(15))

print("[하위 15개 -> Spoke 후보]")
display(stores.sort_values("hub_score", ascending=True)[out_cols].head(15))

print("[외국인 점수 상위 지하철역]")
display(station[["station", "subway_1ticket", "subway_pass", "station_score"]].head(15))

print("[S-DoT 상위 자치구]")
display(sdot_gu.sort_values("sdot_gu_mean", ascending=False).head(10))


[상위 Hub 15개]


,store_name,addr,gu,foreign_500m_sum,sdot_gu_mean,station_match,subway_station_score,dist_olive_m,hub_score,role,inv_multiplier
124,명동본점,서울특별시 중구 명동길 43 (명동1가),Jung-gu,5545.316042,95.603931,명동,15.720436,42.833665,5.933580,Hub_A,2.600000
123,명동역점,서울특별시 중구 퇴계로 134-1(남산동3가),Jung-gu,4255.807917,95.603931,명동,15.720436,87.527171,5.006954,Hub_A,2.574026
100,홍대입구점,서울특별시 마포구 홍익로 25 (서교동),Mapo-gu,2347.296042,81.351896,홍대입구,8.679342,78.139229,2.678807,Hub_A,2.548052
119,서울시청광장점,서울특별시 중구 세종대로 93 (태평로2가),Jung-gu,2660.286831,95.603931,시청,5.525394,16.492814,2.531369,Hub_A,2.522078
97,홍대2호점,서울특별시 마포구 양화로 182 (동교동),Mapo-gu,2193.867120,81.351896,,0.000000,78.180001,1.935120,Hub_A,2.496104
196,롯데마트김포공항점,서울특별시 강서구 하늘길 지하 77 (방화동),Gangseo-gu,2503.211875,45.410175,김포공항,0.063878,775.259450,1.806143,Hub_A,2.470130
120,현대시티아울렛동대문점,서울특별시 중구 장충단로13길 20 (을지로6가),Jung-gu,1983.665246,95.603931,동대문,2.632259,2.347549,1.698728,Hub_A,2.444156
62,종로본점,서울특별시 종로구 삼일대로 395 (종로2가),Jongno-gu,2063.942917,21.369183,,0.000000,69.375929,1.573851,Hub_A,2.418182
121,서울역점,서울특별시 중구 한강대로 405(봉래동2가),Jung-gu,866.838542,95.603931,서울역,14.136158,2.008449,1.450229,Hub_A,2.392208
122,롯데마트서울역점,서울특별시 중구 청파로 426 (봉래동2가),Jung-gu,610.124418,95.603931,서울역,14.136158,301.803095,1.210234,Hub_A,2.366234


[하위 15개 -> Spoke 후보]


,store_name,addr,gu,foreign_500m_sum,sdot_gu_mean,station_match,subway_station_score,dist_olive_m,hub_score,role,inv_multiplier
67,서울양평동점,서울특별시 영등포구 선유로 157 (양평동3가),Yeongdeungpo-gu,72.270625,7.683798,양평,-0.620696,1076.786116,-0.664650,Spoke,0.8
30,구산역점,서울특별시 은평구 서오릉로 156(갈현동),Eunpyeong-gu,57.561232,17.884376,구산,-0.642793,219.341345,-0.639149,Spoke,0.8
149,굿모닝마트유진점,서울특별시 서대문구 통일로 484 (홍제동),Seodaemun-gu,62.391944,6.751534,,0.000000,349.205781,-0.634686,Spoke,0.8
33,응암역점,서울특별시 은평구 은평로 68 (신사동),Eunpyeong-gu,73.590452,17.884376,응암,-0.634115,12.965573,-0.623564,Spoke,0.8
126,효창공원역점,서울특별시 용산구 효창원로 119(용문동),Yongsan-gu,94.037759,4.759628,,0.000000,961.001084,-0.613065,Spoke,0.8
74,당산점,서울특별시 영등포구 양평로 49 (당산동5가),Yeongdeungpo-gu,124.527708,7.683798,당산,-0.430962,110.334448,-0.602675,Spoke,0.8
125,남영역점,서울특별시 용산구 한강대로77길 12 (갈월동),Yongsan-gu,106.978917,4.759628,,0.000000,168.742285,-0.600967,Spoke,0.8
68,롯데마트서울양평점,서울특별시 영등포구 선유로 138 (양평동3가),Yeongdeungpo-gu,142.094792,7.683798,양평,-0.620696,948.640227,-0.599374,Spoke,0.8
69,선유도역점,서울특별시 영등포구 양평로 지하 124(양평동5가),Yeongdeungpo-gu,98.815417,7.683798,,0.000000,39.536473,-0.596908,Spoke,0.8
29,이마트수색점,"서울특별시 은평구 수색로 217 (수색동, 디엠씨 자이1단지)",Eunpyeong-gu,60.861179,17.884376,,0.000000,293.249447,-0.591609,Spoke,0.8


[외국인 점수 상위 지하철역]


,station,subway_1ticket,subway_pass,station_score
78,명동,470471,193.0,15.720436
117,서울역,224303,294.0,14.136158
234,홍대입구,298193,92.0,8.679342
61,동대문역사문화공원(DDP),226653,120.0,8.172208
184,을지로입구,227196,88.0,7.077674
132,시청,133450,99.0,5.525394
202,종로3가,97988,92.0,4.552543
192,잠실(송파구청),118934,77.0,4.465991
106,삼성(무역센터),99681,74.0,3.965469
12,경복궁(정부서울청사),91132,70.0,3.651034


[S-DoT 상위 자치구]


,gu,sdot_gu_mean
8,Guro-gu,173.885596
15,Nowon-gu,162.965618
7,Geumcheon-gu,160.552888
21,Yangcheon-gu,152.747804
1,Dongjak-gu,138.773791
20,Songpa-gu,137.721921
13,Jungnang-gu,129.201571
9,Gwanak-gu,125.942650
0,Dobong-gu,114.791784
12,Jung-gu,95.603931


## 7) 결과 테이블 저장 (선택)

아래 주석 해제 시 저장됩니다.
- `data/pop/hub_spoke_store_scores.csv`


In [ ]:

# save_path = DATA / "pop" / "hub_spoke_store_scores.csv"  # 저장 경로
# stores[out_cols].sort_values("hub_score", ascending=False).to_csv(save_path, index=False, encoding="utf-8-sig")
# print("저장 완료:", save_path)


## 8) 폴리움 시각화

아래 셀은 3가지 지도를 생성합니다.
- 점포 단위 Hub/Spoke 마커 지도
- `hub_score` 가중 히트맵
- 자치구 단위 평균 `hub_score` 코로플레스


In [ ]:

import folium
import branca.colormap as cm
from folium.plugins import HeatMap
from folium.features import GeoJsonTooltip

stores_wgs = stores.to_crs("EPSG:4326").copy()
stores_wgs["lat"] = stores_wgs.geometry.y
stores_wgs["lon"] = stores_wgs.geometry.x
center = [stores_wgs["lat"].mean(), stores_wgs["lon"].mean()]

out_dir = OUTPUT_DIR

# Labels (unicode-escape for stable encoding)
LBL_REGION = "\uad8c\uc5ed"
LBL_ROLE = "\uc5ed\ud560"
LBL_INV = "\uc7ac\uace0\ubc30\uc728"
LBL_FOREIGN = "\ub2e8\uae30\uc678\uad6d\uc778(500m)"
LBL_SDOT = "S-DoT(\uad6c\ud3c9\uade0)"
LBL_COMP = "\uacbd\uc7c1\uac70\ub9ac(\uc62c\ub9ac\ube0c\uc601)"
LBL_LEGEND = "\uc5ed\ud560 \ubc94\ub840"

LAYER_STORE = "Hub/Spoke \uc810\ud3ec"
LAYER_HEAT = "hub_score \ud788\ud2b8\ub9f5"
LAYER_ADMIN = "\uc11c\uc6b8\uc2dc \ud589\uc815\ub3d9 \uacbd\uacc4"
LAYER_ADMIN_WEIGHT = "\ud589\uc815\ub3d9 Hub \uac00\uc911 \uc810\uc218"
LAYER_HUB = "Hub \uc810\ud3ec(Hub_A+Hub_B)"
LAYER_SPOKE = "Spoke \uc810\ud3ec"
LAYER_NORMAL = "Normal \uc810\ud3ec"

# -----------------------------
# Common data setup
# -----------------------------
role_color = {
    "Hub_A": "#d70011",   # 다이소 레드
    "Hub_B": "#ff9800",   # neon green
    "Normal": "#ffd400",  # yellow (usually hidden)
    "Spoke": "#00c2ff",   # cyan
}

adm_dong = gpd.read_file(DATA / "vector" / "BND_ADM_DONG_PG.shp")
adm_dong = adm_dong[adm_dong["ADM_CD"].astype(str).str.startswith("11")].copy().to_crs("EPSG:4326")

# Heatmap weight basis: min-max normalized hub_score per store
vmin = stores_wgs["hub_score"].min()
vmax = stores_wgs["hub_score"].max()
den = (vmax - vmin) if (vmax - vmin) != 0 else 1
weights = (stores_wgs["hub_score"] - vmin) / den
heat_data = [[r.lat, r.lon, w] for r, w in zip(stores_wgs.itertuples(index=False), weights)]

# Admin-dong weighted score: Hub count(70%) + hub_score(30%)
store_for_join = stores_wgs[["store_id", "role", "hub_score", "geometry"]].copy()
join_dong = gpd.sjoin(store_for_join, adm_dong[["ADM_CD", "ADM_NM", "geometry"]], how="left", predicate="within")

dong_stats = join_dong.groupby(["ADM_CD", "ADM_NM"], as_index=False).agg(
    store_cnt=("store_id", "count"),
    hubA_cnt=("role", lambda s: int((s == "Hub_A").sum())),
    hubB_cnt=("role", lambda s: int((s == "Hub_B").sum())),
    hub_score_mean=("hub_score", "mean"),
)

dong_stats["hub_weighted_cnt"] = dong_stats["hubA_cnt"] * 2 + dong_stats["hubB_cnt"]

adm_dong_metric = adm_dong.merge(dong_stats, on=["ADM_CD", "ADM_NM"], how="left")
for c in ["store_cnt", "hubA_cnt", "hubB_cnt", "hub_score_mean", "hub_weighted_cnt"]:
    adm_dong_metric[c] = adm_dong_metric[c].fillna(0)


def minmax_norm(series):
    s = series.astype(float)
    mn, mx = s.min(), s.max()
    if mx == mn:
        return s * 0
    return (s - mn) / (mx - mn)


adm_dong_metric["hub_cnt_norm"] = minmax_norm(adm_dong_metric["hub_weighted_cnt"])
adm_dong_metric["hub_score_norm"] = minmax_norm(adm_dong_metric["hub_score_mean"])
adm_dong_metric["dong_priority"] = 0.7 * adm_dong_metric["hub_cnt_norm"] + 0.3 * adm_dong_metric["hub_score_norm"]


PANE_ADMIN = "pane_admin_bottom"
PANE_ADMIN_WEIGHT = "pane_admin_weight"
PANE_HEAT = "pane_heat_mid"
PANE_STORE = "pane_store_top"


def ensure_layer_panes(m):
    if PANE_ADMIN not in m._children:
        folium.map.CustomPane(PANE_ADMIN, z_index=320).add_to(m)
    if PANE_ADMIN_WEIGHT not in m._children:
        folium.map.CustomPane(PANE_ADMIN_WEIGHT, z_index=330).add_to(m)
    if PANE_HEAT not in m._children:
        folium.map.CustomPane(PANE_HEAT, z_index=360).add_to(m)
    if PANE_STORE not in m._children:
        folium.map.CustomPane(PANE_STORE, z_index=650).add_to(m)


def add_store_role_layers(m, show_hub=True, show_spoke=True, show_normal=False):
    ensure_layer_panes(m)

    fg_hub = folium.FeatureGroup(name=LAYER_HUB, show=show_hub)
    fg_spoke = folium.FeatureGroup(name=LAYER_SPOKE, show=show_spoke)
    fg_normal = folium.FeatureGroup(name=LAYER_NORMAL, show=show_normal)

    for _, r in stores_wgs.sort_values("hub_score", ascending=False).iterrows():
        color = role_color.get(r["role"], "#888888")
        radius = float(max(4, min(12, 4 + r["inv_multiplier"] * 2)))
        popup_html = (
            f"<b>{r['store_name']}</b><br>"
            f"{LBL_REGION}: {r['gu']}<br>"
            f"{LBL_ROLE}: {r['role']}<br>"
            f"hub_score: {r['hub_score']:.3f}<br>"
            f"{LBL_INV}: {r['inv_multiplier']:.2f}x<br>"
            f"{LBL_FOREIGN}: {r['foreign_500m_sum']:.1f}<br>"
            f"{LBL_SDOT}: {r['sdot_gu_mean']:.1f}<br>"
            f"{LBL_COMP}: {r['dist_olive_m']:.1f}m"
        )
        marker = folium.CircleMarker(
            location=[r["lat"], r["lon"]],
            radius=radius + 1,
            color="#111111",
            fill=True,
            fill_color=color,
            fill_opacity=0.95,
            opacity=1.0,
            weight=2.2,
            pane=PANE_STORE,
            tooltip=f"{r['store_name']} | {r['role']}",
            popup=folium.Popup(popup_html, max_width=320),
        )

        role = r["role"]
        if role in ("Hub_A", "Hub_B"):
            marker.add_to(fg_hub)
        elif role == "Spoke":
            marker.add_to(fg_spoke)
        else:
            marker.add_to(fg_normal)

    fg_hub.add_to(m)
    fg_spoke.add_to(m)
    fg_normal.add_to(m)
    return {"hub": fg_hub, "spoke": fg_spoke, "normal": fg_normal}


def add_heat_layer(m, show=True):
    ensure_layer_panes(m)
    fg = folium.FeatureGroup(name=LAYER_HEAT, show=show)
    HeatMap(
        heat_data,
        radius=18,
        blur=14,
        max_zoom=14,
        min_opacity=0.35,
        pane=PANE_HEAT,
    ).add_to(fg)
    fg.add_to(m)
    return fg


def add_admin_boundary_layer(m, show=True):
    ensure_layer_panes(m)
    fg = folium.FeatureGroup(name=LAYER_ADMIN, show=show)
    folium.GeoJson(
        adm_dong,
        pane=PANE_ADMIN,
        style_function=lambda _: {
            "fillColor": "#b5b5b5",
            "fillOpacity": 0.03,
            "color": "#7d7d7d",
            "weight": 0.35,
            "opacity": 0.4,
        },
        tooltip=GeoJsonTooltip(
            fields=["ADM_NM"],
            aliases=["\ud589\uc815\ub3d9"],
            localize=True,
            sticky=False,
        ),
    ).add_to(fg)
    fg.add_to(m)
    return fg


def add_admin_weight_layer(m, show=False):
    ensure_layer_panes(m)
    fg = folium.FeatureGroup(name=LAYER_ADMIN_WEIGHT, show=show)

    vmin_w = float(adm_dong_metric["dong_priority"].min())
    vmax_w = float(adm_dong_metric["dong_priority"].max())
    cmap = cm.linear.YlOrRd_09.scale(vmin_w, vmax_w)

    def style_fn(feature):
        val = feature["properties"].get("dong_priority", 0)
        return {
            "fillColor": cmap(val),
            "fillOpacity": 0.52,
            "color": "#5a5a5a",
            "weight": 0.45,
            "opacity": 0.5,
        }

    folium.GeoJson(
        adm_dong_metric,
        pane=PANE_ADMIN_WEIGHT,
        style_function=style_fn,
        tooltip=GeoJsonTooltip(
            fields=["ADM_NM", "hubA_cnt", "hubB_cnt", "hub_weighted_cnt", "hub_score_mean", "dong_priority"],
            aliases=[
                "\ud589\uc815\ub3d9",
                "Hub_A \uc218",
                "Hub_B \uc218",
                "Hub \uac00\uc911\uce58(2*A+1*B)",
                "\ud3c9\uade0 hub_score",
                "\ud589\uc815\ub3d9 \uc6b0\uc120\uc810\uc218",
            ],
            localize=True,
            sticky=False,
        ),
    ).add_to(fg)

    fg.add_to(m)
    cmap.caption = "\ud589\uc815\ub3d9 Hub \uac00\uc911 \uc810\uc218 (0.7*Hub\uac1c\uc218 + 0.3*\uc810\uc218)"
    cmap.add_to(m)
    return fg


def add_role_legend(m):
    legend_html = f"""
    <div style=\"position: fixed; bottom: 20px; left: 20px; width: 170px;
                background: white; z-index:9999; border:2px solid #555;
                border-radius:8px; padding:10px; font-size:13px;\">
    <b>{LBL_LEGEND}</b><br>
    <span style=\"color:#ff00ff;\">&#9679;</span> Hub_A<br>
    <span style=\"color:#39ff14;\">&#9679;</span> Hub_B<br>
    <span style=\"color:#ffd400;\">&#9679;</span> Normal<br>
    <span style=\"color:#00c2ff;\">&#9679;</span> Spoke
    </div>
    """
    m.get_root().html.add_child(folium.Element(legend_html))

# 1) Store role map (+ heatmap toggle)
m_store = folium.Map(location=center, zoom_start=11, tiles="CartoDB Positron")
add_admin_boundary_layer(m_store, show=True)
add_heat_layer(m_store, show=False)
add_store_role_layers(m_store, show_hub=True, show_spoke=True, show_normal=False)
add_role_legend(m_store)
folium.LayerControl(collapsed=False).add_to(m_store)
store_map_path = out_dir / "hub_spoke_store_map.html"
m_store.save(store_map_path)

# 2) Heatmap-only + boundary
m_heat_only = folium.Map(location=center, zoom_start=11, tiles="CartoDB Positron")
add_admin_boundary_layer(m_heat_only, show=True)
add_heat_layer(m_heat_only, show=True)
folium.LayerControl(collapsed=False).add_to(m_heat_only)
heat_only_path = out_dir / "hub_score_heatmap_only.html"
m_heat_only.save(heat_only_path)

# 3) Combined layered map (requested)
m_combo = folium.Map(location=center, zoom_start=11, tiles="CartoDB Positron")
add_admin_boundary_layer(m_combo, show=True)
add_admin_weight_layer(m_combo, show=False)
add_heat_layer(m_combo, show=True)
add_store_role_layers(m_combo, show_hub=True, show_spoke=True, show_normal=False)
add_role_legend(m_combo)
folium.LayerControl(collapsed=False).add_to(m_combo)

combo_path_main = out_dir / "hub_score_heatmap.html"
m_combo.save(combo_path_main)

combo_path_alt = out_dir / "hub_layered_map.html"
m_combo.save(combo_path_alt)

# 4) Dedicated admin-dong weighted map
m_dong = folium.Map(location=center, zoom_start=11, tiles="CartoDB Positron")
add_admin_boundary_layer(m_dong, show=True)
add_admin_weight_layer(m_dong, show=True)
folium.LayerControl(collapsed=False).add_to(m_dong)
admin_weight_path = out_dir / "admin_dong_hub_weight_map.html"
m_dong.save(admin_weight_path)

# 5) GU-level choropleth (existing)
adm = gpd.read_file(DATA / "vector" / "BND_ADM_DONG_PG.shp")
adm = adm[adm["ADM_CD"].astype(str).str.startswith("11")].copy()
adm["gu_code"] = adm["ADM_CD"].astype(str).str[:5]
adm_gu = adm.dissolve(by="gu_code", as_index=False)

code_to_gu = {
    "11110": "Jongno-gu", "11140": "Jung-gu", "11170": "Yongsan-gu",
    "11200": "Seongdong-gu", "11215": "Gwangjin-gu", "11230": "Dongdaemun-gu",
    "11260": "Jungnang-gu", "11290": "Seongbuk-gu", "11305": "Gangbuk-gu",
    "11320": "Dobong-gu", "11350": "Nowon-gu", "11380": "Eunpyeong-gu",
    "11410": "Seodaemun-gu", "11440": "Mapo-gu", "11470": "Yangcheon-gu",
    "11500": "Gangseo-gu", "11530": "Guro-gu", "11545": "Geumcheon-gu",
    "11560": "Yeongdeungpo-gu", "11590": "Dongjak-gu", "11620": "Gwanak-gu",
    "11650": "Seocho-gu", "11680": "Gangnam-gu", "11710": "Songpa-gu",
    "11740": "Gangdong-gu",
}
adm_gu["gu"] = adm_gu["gu_code"].map(code_to_gu)

gu_stats = stores.groupby("gu", as_index=False).agg(
    store_cnt=("store_id", "count"),
    hub_score_mean=("hub_score", "mean"),
    hubA_cnt=("role", lambda s: int((s == "Hub_A").sum())),
    hubB_cnt=("role", lambda s: int((s == "Hub_B").sum())),
)

adm_gu = adm_gu.merge(gu_stats, on="gu", how="left")
for c in ["store_cnt", "hub_score_mean", "hubA_cnt", "hubB_cnt"]:
    adm_gu[c] = adm_gu[c].fillna(0)

adm_gu = adm_gu.to_crs("EPSG:4326")

m_gu = folium.Map(location=center, zoom_start=11, tiles="CartoDB Positron")
folium.Choropleth(
    geo_data=adm_gu,
    data=adm_gu,
    columns=["gu_code", "hub_score_mean"],
    key_on="feature.properties.gu_code",
    fill_color="YlOrRd",
    fill_opacity=0.75,
    line_opacity=0.3,
    nan_fill_opacity=0.1,
    legend_name="\uc790\uce58\uad6c \ud3c9\uade0 hub_score",
).add_to(m_gu)

folium.GeoJson(
    adm_gu,
    style_function=lambda _: {"fillOpacity": 0, "color": "#666", "weight": 0.6},
    tooltip=GeoJsonTooltip(
        fields=["gu", "store_cnt", "hubA_cnt", "hubB_cnt", "hub_score_mean"],
        aliases=["\uc790\uce58\uad6c", "\uc810\ud3ec\uc218", "Hub_A \uc218", "Hub_B \uc218", "\ud3c9\uade0 hub_score"],
        localize=True,
        sticky=False,
    ),
).add_to(m_gu)

gu_map_path = out_dir / "district_hub_score_map.html"
m_gu.save(gu_map_path)

print("\uc800\uc7a5 \uc644\ub8cc:")
print(" -", store_map_path)
print(" -", heat_only_path)
print(" -", combo_path_main, "(\ud1b5\ud569 \ub808\uc774\uc5b4)")
print(" -", combo_path_alt)
print(" -", admin_weight_path)
print(" -", gu_map_path)

저장 완료:
 - G:\Final_proj\daiso\GIS\hub_spoke_store_map.html
 - G:\Final_proj\daiso\GIS\hub_score_heatmap_only.html
 - G:\Final_proj\daiso\GIS\hub_score_heatmap.html (통합 레이어)
 - G:\Final_proj\daiso\GIS\hub_layered_map.html
 - G:\Final_proj\daiso\GIS\admin_dong_hub_weight_map.html
 - G:\Final_proj\daiso\GIS\district_hub_score_map.html


### Q1. 왜 장기체류 외국인은 빼나?
- 본 모델은 관광·단기체류 수요의 급등 리스크를 방어하는 것이 목적이므로 의도적으로 제외했습니다.

### Q2. 지표 가중치는 왜 0.55/0.20/0.15/0.10인가?
- 품절 발생에 직접적으로 영향을 주는 “점포 주변 단기외국인 수요”를 최우선으로 두고, 광역 유동·역세권 신호·경쟁 압력 순으로 배치했습니다.

### Q3. 이 모델을 매주 운영할 수 있나?
- 가능합니다. 데이터 갱신 주기를 맞추면 점포별 role과 재고배율을 자동 업데이트하는 배치 파이프라인으로 확장 가능합니다.


## 15) 장표 2(31-C)용 정량 시각화 생성

아래 코드는 장표 2에 바로 붙일 수 있는 정량 시각화 PNG를 생성합니다.
- `slide31_kpi_before_after.png`: 수요충족률/미충족수요 Before-After
- `slide31_share_compare.png`: 점포비중/수요비중/최적배분비중 비교
- `slide31_scenario_sales_profit.png`: 보수/기준/낙관 시나리오 매출·수익
- `slide31_top_store_cumshare.png`: 상위 점포 누적 수요 기여도



In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

asset_dir = OUTPUT_DIR / "slide31_assets"
asset_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 1) KPI Before-After
# -----------------------------
before_fill = 55.0
after_fill = 64.7
before_unmet = 34080
after_unmet = 26757

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), dpi=150)

axes[0].bar(["Before", "After"], [before_fill, after_fill], color=["#9ca3af", "#2563eb"])
axes[0].set_title("수요충족률 개선")
axes[0].set_ylabel("수요충족률(%)")
axes[0].set_ylim(0, 80)
for i, v in enumerate([before_fill, after_fill]):
    axes[0].text(i, v + 1.2, f"{v:.1f}%", ha="center", fontsize=10, fontweight="bold")

axes[1].bar(["Before", "After"], [before_unmet, after_unmet], color=["#9ca3af", "#ea580c"])
axes[1].set_title("미충족 수요 감소")
axes[1].set_ylabel("미충족 수요")
for i, v in enumerate([before_unmet, after_unmet]):
    axes[1].text(i, v + 700, f"{v:,.0f}", ha="center", fontsize=10, fontweight="bold")

fig.suptitle("동일 총재고 기준 Before-After", fontsize=13, fontweight="bold")
fig.tight_layout()
path1 = asset_dir / "slide31_kpi_before_after.png"
fig.savefig(path1, bbox_inches="tight")
plt.close(fig)

# -----------------------------
# 2) Share comparison by role
# -----------------------------
role_order = ["Hub_A", "Hub_B", "Normal", "Spoke"]
role_stats = stores.groupby("role", as_index=False).agg(
    store_cnt=("store_id", "count"),
    demand_sum=("foreign_500m_sum", "sum"),
    alloc_weight=("inv_multiplier", "sum"),
)
role_stats["role"] = pd.Categorical(role_stats["role"], categories=role_order, ordered=True)
role_stats = role_stats.sort_values("role")

role_stats["store_share"] = role_stats["store_cnt"] / role_stats["store_cnt"].sum()
role_stats["demand_share"] = role_stats["demand_sum"] / role_stats["demand_sum"].sum()
role_stats["alloc_share"] = role_stats["alloc_weight"] / role_stats["alloc_weight"].sum()

x = np.arange(len(role_stats))
w = 0.24
fig, ax = plt.subplots(figsize=(10.8, 4.4), dpi=150)
ax.bar(x - w, role_stats["store_share"] * 100, width=w, label="점포 비중", color="#9ca3af")
ax.bar(x, role_stats["demand_share"] * 100, width=w, label="수요 비중", color="#2563eb")
ax.bar(x + w, role_stats["alloc_share"] * 100, width=w, label="최적배분 비중", color="#ea580c")

ax.set_xticks(x)
ax.set_xticklabels(role_stats["role"].astype(str))
ax.set_ylabel("비중(%)")
ax.set_title("역할군별 점포/수요/배분 비중 비교")
ax.legend(frameon=False)
ax.grid(axis="y", linestyle="--", alpha=0.3)

for i, v in enumerate(role_stats["alloc_share"] * 100):
    ax.text(i + w, v + 0.6, f"{v:.1f}", ha="center", fontsize=9)

fig.tight_layout()
path2 = asset_dir / "slide31_share_compare.png"
fig.savefig(path2, bbox_inches="tight")
plt.close(fig)

# -----------------------------
# 3) Scenario sales/profit
# -----------------------------
scn = pd.DataFrame({
    "시나리오": ["보수", "기준", "낙관"],
    "추가매출(억)": [84, 169, 281],
    "수익개선(억)": [29, 59, 98],
})

x = np.arange(len(scn))
w = 0.35
fig, ax = plt.subplots(figsize=(9.6, 4.6), dpi=150)
ax.bar(x - w/2, scn["추가매출(억)"], width=w, label="추가매출(억)", color="#2563eb")
ax.bar(x + w/2, scn["수익개선(억)"], width=w, label="수익개선(억)", color="#ea580c")

ax.set_xticks(x)
ax.set_xticklabels(scn["시나리오"])
ax.set_ylabel("억원")
ax.set_title("시나리오별 추가매출/수익개선")
ax.legend(frameon=False)
ax.grid(axis="y", linestyle="--", alpha=0.3)

for i, (s, p) in enumerate(zip(scn["추가매출(억)"], scn["수익개선(억)"])):
    ax.text(i - w/2, s + 4, f"{s}", ha="center", fontsize=9)
    ax.text(i + w/2, p + 2, f"{p}", ha="center", fontsize=9)

fig.tight_layout()
path3 = asset_dir / "slide31_scenario_sales_profit.png"
fig.savefig(path3, bbox_inches="tight")
plt.close(fig)

# -----------------------------
# 4) Cumulative demand contribution by top stores
# -----------------------------
top = stores.sort_values("foreign_500m_sum", ascending=False).reset_index(drop=True).copy()
top["rank"] = np.arange(1, len(top) + 1)
top["cum_share"] = top["foreign_500m_sum"].cumsum() / top["foreign_500m_sum"].sum() * 100

fig, ax = plt.subplots(figsize=(10.8, 4.4), dpi=150)
ax.plot(top["rank"], top["cum_share"], color="#2563eb", linewidth=2)
for k in [10, 20, 30, 50]:
    if k <= len(top):
        y = float(top.loc[top["rank"] == k, "cum_share"].iloc[0])
        ax.scatter([k], [y], color="#ea580c", s=28, zorder=3)
        ax.text(k, y + 1.0, f"Top{k}: {y:.1f}%", ha="center", fontsize=9)

ax.set_xlabel("상위 점포 순위")
ax.set_ylabel("누적 수요 기여도(%)")
ax.set_title("상위 점포 누적 수요 기여도")
ax.set_ylim(0, 100)
ax.grid(alpha=0.3, linestyle="--")

fig.tight_layout()
path4 = asset_dir / "slide31_top_store_cumshare.png"
fig.savefig(path4, bbox_inches="tight")
plt.close(fig)

print("장표 자산 저장 완료:")
print(" -", path1)
print(" -", path2)
print(" -", path3)
print(" -", path4)



장표 자산 저장 완료:
 - G:\Final_proj\daiso\GIS\slide31_assets\slide31_kpi_before_after.png
 - G:\Final_proj\daiso\GIS\slide31_assets\slide31_share_compare.png
 - G:\Final_proj\daiso\GIS\slide31_assets\slide31_scenario_sales_profit.png
 - G:\Final_proj\daiso\GIS\slide31_assets\slide31_top_store_cumshare.png


1. 핵심 문제
- 내국인(테스트형) vs 외국인(하울형)
  수요 구조가 다른데 재고 정책이 동일

2. 전체 매장 재고 균등배분시 55.0%의 낮은 수요충족률
그러므로 상권 기반 Hub & Spoke 물류 이원화 필요

3. “몇 배”의 숫자 근거 (수요 대비)

주요 변수
1. foreign_500m_sum
점포 반경 500m 안의 250m 격자 단기체류 외국인 수요 합
Σ(fs_cell_mean), predicate=within(buffer 500m)
2. subway_station_score
외국인 지하철 이용 강도 점수
z(1회권 이용량) + z(기간권 이용량)
3. sdot_gu_mean
점포가 속한 자치구의 S-DoT 유동인구 평균
mean(visitors | gu)
4. comp_50_100
경쟁 로드숍 최근접 거리가 50~100m이면 1, 아니면 0
- 계산: 1(50 <= dist_olive_m <= 100), else 0

의사결정 규칙 1: 점수와 역할
수요밀도 점수(hub_score)
hub_score = 0.55*z(foreign_500m_sum)
 + 0.20*z(sdot_gu_mean)
 + 0.15*z(subway_station_score)
 + 0.10*z(comp_50_100)

역할 분류 규칙
- Hub_A: hub_score >= P90
- Hub_B: P70 <= hub_score < P90
- Spoke: hub_score < P30
- Normal: 그 외

`hub_score` 구간별 점포 주변 단기외국인 수요 중앙값은 다음과 같습니다.

- P50~70 구간 중앙값: **137.63** (기준구간)
- P70~90 구간 중앙값: **323.81**  -> 기준 대비 **2.35배**
- P90~100 구간 중앙값: **939.15** -> 기준 대비 **6.82배**
- P0~30 구간 중앙값: **90.43**  -> 기준 대비 **0.66배**

이 수요차를 그대로 적용하면 과적재고 리스크가 크기 때문에, 운영 캡을 두어 실행 배율을 다음과 같이 정의

- **Hub_A: 2.0~2.6x** (상위 10% 구간, 품절 방어 우선)
- **Hub_B: 1.3~1.8x** (상위 30~10% 구간, 수요 증가 흡수)
- **Normal: 1.0x**
- **Spoke: 0.8x** (하위 30% 구간, 과적재고 방지)

4. 실제 그룹별 집계(현재 결과)
| Role | 점포수 | 단기외국인 중앙값(500m) | Normal 대비 비율 | 적용 배율 범위 |
|---|---:|---:|---:|---:|
| Hub_A | 24 | 936.46 | 7.02x | 2.00~2.60x |
| Hub_B | 46 | 316.04 | 2.37x | 1.30~1.79x |
| Normal | 92 | 133.49 | 1.00x | 1.00x |
| Spoke | 69 | 90.44 | 0.68x | 0.80x |

## ==========================================================

### 왜 물류 이원화가 필요한가 + 기준이 무엇인가

- 상단 15%: 제목 + 한 줄 결론
- 중단 왼쪽 55%: 문제정의 카드(현 상태 KPI)
- 중단 오른쪽 45%: 산정 기준 카드(변수/점수식/역할/배율)
- 하단 20%: 실제 그룹 집계 표

```text
31. 왜 Hub & Spoke 물류 이원화가 필요한가
같은 재고를 뿌려도, 수요가 몰리는 점포에서 결품이 나면 손실이 커집니다.
```

```text
[핵심 문제]
- 내국인(테스트형) vs 외국인(하울형)
  수요 구조가 다른데 재고 정책은 동일

[균등배분 시 현 상태]
- 수요충족률: 55.0%
- 미충족 수요: 34,080

[해석]
수요가 몰리는 관광 상권에서 먼저 품절이 발생하고,
그 순간 판매 실패가 아니라 기회손실이 됩니다.
```

```text
[주요 변수]
1) foreign_500m_sum: 점포 반경 500m 단기체류 외국인 수요 합
2) subway_station_score: 외국인 지하철 이용 강도 점수
3) sdot_gu_mean: 자치구 S-DoT 유동인구 평균
4) comp_50_100: 경쟁점 거리 50~100m 플래그

[의사결정 규칙 1: 점수와 역할]
hub_score = 0.55*z(foreign_500m_sum)
          + 0.20*z(sdot_gu_mean)
          + 0.15*z(subway_station_score)
          + 0.10*z(comp_50_100)

- Hub_A: hub_score >= P90
- Hub_B: P70 <= hub_score < P90
- Spoke: hub_score < P30
- Normal: 그 외

[의사결정 규칙 2: 운영 배율]
- Hub_A: 2.0~2.6x
- Hub_B: 1.3~1.8x
- Normal: 1.0x
- Spoke: 0.8x
```

| Role | 점포수 | 단기외국인 중앙값(500m) | Normal 대비 비율 | 적용 배율 범위 |
|---|---:|---:|---:|---:|
| Hub_A | 24 | 936.46 | 7.02x | 2.00~2.60x |
| Hub_B | 46 | 316.04 | 2.37x | 1.30~1.79x |
| Normal | 92 | 133.49 | 1.00x | 1.00x |
| Spoke | 69 | 90.44 | 0.68x | 0.80x |

---

### GIS 시각화로 어디에 어떤 액션이 필요한지 바로 보여주기

- Heat ON, Hub ON, Spoke ON, Normal OFF
- 행정동 경계 ON
- 중심/줌 고정 (발표 리허설 화면과 동일)

| 순위 | 매장명 | 권역 | 역할 | hub_score | 권장 배율 |
|---:|---|---|---|---:|---:|
| 1 | 명동본점 | Jung-gu | Hub_A | 5.934 | 2.60x |
| 2 | 명동역점 | Jung-gu | Hub_A | 5.007 | 2.57x |
| 3 | 홍대입구점 | Mapo-gu | Hub_A | 2.679 | 2.55x |
| 4 | 서울시청광장점 | Jung-gu | Hub_A | 2.531 | 2.52x |
| 5 | 홍대2호점 | Mapo-gu | Hub_A | 1.935 | 2.50x |

```text
[권역 액션]
- Hub(명동·홍대·강남): 상위 SKU 재고 확대
- Spoke: 신제품 구색 확대, 저회전 SKU 압축

[운영 포인트]
- Hub는 품절률 KPI 우선
- Spoke는 신제품 회전률 KPI 우선
```

---

### 효율화를 어떻게 수익으로 만들었는지 한눈에 보여주기
```text
[동일 총재고 기준 성과]
- 수요충족률: 55.0% -> 64.7% (+9.66%p)
- 미충족 수요: 34,080 -> 26,757 (-21.5%)

[연간 환산 (2025 뷰티 매출 4,000억)]
- 추가매출: 84억~281억 (기준 169억)
- 수익개선: 29억~98억 (기준 59억, GM 35%)

[결론]
효율화는 단순 물류 개선이 아니라
결품 손실을 줄여 이익을 방어하는 전략이다.
```
